# Chapter 5.5: Vector databases from scratch

> **Learn with Prof Rod** — *Build Your Always-On AI Agent From Scratch*.
> **Read the full book and get the latest learning materials:** [https://profrod.ai/book](https://profrod.ai/book).
> **Join the Prof Rod learner community:** [https://profrod.ai/community](https://profrod.ai/community)
> — bring your questions, compare experiments and share what you build.
> **Original source and updates:** [profrodai/sovereign-agent](https://github.com/profrodai/sovereign-agent).

**Student edition · interlude after Chapter 5 · the ITAM class of 2026-09-24**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/profrodai/sovereign-agent/blob/main/interludes/ch05-5-vector-databases/profrod-sovereign-agent-ch05-5-vector-databases-exercise.ipynb) Runs on Google Colab as it ships today, or on any local Python 3.12+ kernel.

Chapter 5 gave Lucy's agent a durable memory and selected from it by session, revision and word overlap, with no embeddings. This interlude builds what the book leaves out: embeddings and a vector database, from scratch, then the same with Chroma.

By the end you should be able to:
1. Turn a word into a one-hot vector and look up its embedding with a matrix product.
2. Explain how training moves word vectors so that neighbors point the same way.
3. Build one vector per note: look up, transform, pool, normalize.
4. Store vectors as rows, rank them with one matrix product, and do the same with **Chroma** and real embeddings.

| Part | In class | In this notebook |
|---|---|---|
| 1 | A word is not a number | one-hot, lookup, your turn 1 |
| 2 | Learning where words live | training pairs, one step, training, cosine, your turn 2 |
| 3 | One vector per note | by hand, pooling, your turn 3 |
| 4 | Store and retrieve | our store, the stale note, your turn 4, **then Chroma on Colab** |

The notes are constructed for teaching and match the live class; every number comes from running the code below.

## Run the setup cell first

It writes the class lab (`vector_db_lab.py`, Python standard library only) into a temporary folder and imports it. Nothing is installed and no key is needed until Part 4b.

<details><summary>Setup cell (supplied; you don't need to read it)</summary>

In [ ]:
import base64, hashlib, importlib, os, sys, tempfile
if sys.version_info[:2] < (3, 12):
    raise RuntimeError("This notebook needs Python 3.12 or newer, which is what Google Colab runs today.")
LAB_SHA256 = "cad48ddf2b48025f729b5d521c953502b9e2dcaaef0360a043ff669a4fee9a83"
LAB_B64 = "..."
lab_bytes = base64.b64decode(LAB_B64)
if hashlib.sha256(lab_bytes).hexdigest() != LAB_SHA256:
    raise RuntimeError("The embedded lab does not match its checksum; download the notebook again.")
LAB_DIR = tempfile.mkdtemp(prefix="vector-db-class-")
with open(os.path.join(LAB_DIR, "vector_db_lab.py"), "wb") as handle:
    handle.write(lab_bytes)
sys.path.insert(0, LAB_DIR)
import vector_db_lab
importlib.reload(vector_db_lab)
from vector_db_lab import *
print("Setup ready: vector_db_lab", LAB_SHA256[:12], "| notes:", len(NOTES))

</details>

## Part 1 · A word is not a number

**Prediction:** if we number the words in sorted order, will "vanilla" be closer to "vegan" or to "chocolate"? Write it down, then run.

In [ ]:
vocab = build_vocab(NOTES)
print(len(vocab), "words:", " ".join(vocab))
for word in ["chocolate", "mango", "sorbet", "vanilla", "vegan"]:
    print(f"{word:>10} -> id {vocab.index(word)}")

The ids come from the alphabet, so "vanilla" (21) sits next to "vegan" (22) and far from "chocolate" (1). A one-hot vector makes no such claim — and no other claim either:

In [ ]:
mini = ["chocolate", "vanilla", "vegan"]
for a in mini:
    for b in mini:
        if a < b:
            print(f"one-hot({a}) · one-hot({b}) =", dot(one_hot(a, mini), one_hot(b, mini)))

### The embedding lookup is a matrix product

Multiplying the table `E` (one column per word) by a one-hot vector returns that word's column.

In [ ]:
vocab, E, history, losses = train(NOTES)
x = one_hot("vanilla", vocab)
table = [[E[w][k] for w in vocab] for k in range(2)]
print([round(v, 2) for v in matvec(table, x)])
print([round(v, 2) for v in E["vanilla"]])

### Your turn 1 (3 minutes)

**Predict first:** is "mango" closer to "vegan" or to "invoice"? Then run.

In [ ]:
print(E["mango"])
print(round(cosine(E["mango"], E["vegan"]), 2))
print(round(cosine(E["mango"], E["invoice"]), 2))

## Part 2 · Learning where words live

Training data: slide a window of two words over each note and pair every word with its neighbors.

In [ ]:
pairs = training_pairs([NOTES[0]])
print(NOTES[0])
for center, context in pairs:
    print(f"  ({center}, {context})")
print(len(training_pairs(NOTES)), "pairs from all 12 notes")

One real training step, taken from the lab's run after epoch 5: score a real pair and a random word, compare with the targets, nudge the vector.

In [ ]:
step = one_step()
for row in step["rows"]:
    print(f"{row['word']:>8}: score {row['score']:+.2f}  sigmoid {row['p']:.2f}  target {row['label']}  error {row['g']:+.2f}")
print("E[vegan]", [round(v, 2) for v in step["e"]], "->", [round(v, 2) for v in step["new_e"]])

Now 120 epochs. **Prediction:** which words will end up pointing the same way as "vegan"?

In [ ]:
show = ["vanilla", "chocolate", "cone", "kids", "vegan", "dairy-free", "sorbet", "mango", "supplier", "invoice", "tubs"]
try:
    import matplotlib.pyplot as plt
    figure, axes = plt.subplots(1, len(history), figsize=(4 * len(history), 4), sharex=True, sharey=True)
    for ax, (epoch, snapshot) in zip(axes, sorted(history.items())):
        for word in show:
            x0, y0 = snapshot[word]
            ax.scatter(x0, y0)
            ax.annotate(word, (x0, y0), fontsize=8)
        ax.set_title(f"epoch {epoch}")
        ax.axhline(0, linewidth=0.5); ax.axvline(0, linewidth=0.5)
    plt.show()
except ImportError:
    for epoch, snapshot in sorted(history.items()):
        print("epoch", epoch, {w: [round(v, 2) for v in snapshot[w]] for w in show[:5]})
print(f"loss per pair: {losses[0]:.3f} -> {losses[-1]:.3f}")

Cosine compares directions; squared L2 (Chroma's default distance) also sees length. Normalize first and they agree.

In [ ]:
a, b = E["vegan"], [2 * v for v in E["vegan"]]
print("vegan vs 2×vegan: cos", round(cosine(a, b), 2), " squared L2", round(DISTANCES["l2"](a, b), 2))
print("vegan vs dairy-free: cos", round(cosine(a, E["dairy-free"]), 2), " squared L2", round(DISTANCES["l2"](a, E["dairy-free"]), 2))
print("normalize([1, 2, 1]) =", [round(v, 2) for v in normalize([1, 2, 1])])

### Your turn 2 (3 minutes)

**Predict first:** order these three pairs from highest cosine to lowest. Then run.

In [ ]:
for a, b in [("vegan", "dairy-free"), ("vegan", "invoice"), ("vanilla", "chocolate")]:
    print(a, b, round(cosine(E[a], E[b]), 2))

## Part 3 · One vector per note

By hand, after Tom Yeh's "AI by Hand": hand-set integers so every step fits in your head. Look up the word columns, apply one layer `ReLU(W·x + b)`, then average each row.

In [ ]:
def show_matrix(name, rows):
    print(name)
    for row in rows:
        print("   ", "  ".join(f"{v:>5}" if isinstance(v, int) else f"{v:>5.2f}" for v in row))

stages = by_hand("oat vanilla vegan")
show_matrix("word columns X", stages["X"])
show_matrix("W·X + b", stages["pre"])
show_matrix("ReLU", stages["H"])
print("mean pooling:", [round(v, 3) for v in stages["pooled"]])

### Your turn 3 (3 minutes, paper)

Pool **"vegan oat kids"** by hand with the same table, `W` and `b` (`HAND_WORDS`, `HAND_W`, `HAND_B`). Write four numbers, then check:

<details><summary>Check your answer</summary>

In [ ]:
print(HAND_WORDS["vegan"], HAND_WORDS["oat"], HAND_WORDS["kids"])
print([round(v, 3) for v in by_hand("vegan oat kids")["pooled"]])

</details>

Mean pooling forgets word order — "kids oat vegan" gives the same vector:

In [ ]:
print(by_hand("vegan oat kids")["pooled"] == by_hand("kids oat vegan")["pooled"])

The lab's real note vectors: the mean of the learned word vectors, normalized to length 1.

In [ ]:
vocab, E, store = lucy_store(space="cosine")
for note_id, row in store.rows.items():
    print(note_id, [round(v, 2) for v in row["embedding"]], row["document"])

## Part 4 · Store and retrieve

Search by hand: the stored rows times the query vector give one score per note.

In [ ]:
stored, query_vector, scores, best = by_hand_search()
for note_id, score in scores.items():
    print(note_id, f"{round(score * 9)}/9", HAND_NOTES[note_id])
print("nearest:", best)

Our from-scratch store, with the same verbs Chroma uses. **Prediction:** will a note that never says "vegan" come back for "a vegan treat"?

In [ ]:
print("keyword search:", keyword_search("a vegan treat", NOTES))
hits = store.query(["a vegan treat"], n_results=4)
print(hits["ids"])
print(hits["documents"][0])

The nearest note can be wrong: the delivery day moved to Friday, and the old Monday note still wins until a filter or a delete says otherwise.

In [ ]:
before, filtered, deleted = stale_memory_demo()
print("nearest two:", before["documents"][0])
print("where current=True:", filtered["documents"][0])
print("after delete:", deleted["documents"][0])
for ef in (1, 3):
    walk = graph_search(store, "a vegan treat", entry="n12", ef=ef, links=1)
    print(f"graph search ef={ef}: found {walk['found']} after {walk['comparisons']} of 12 comparisons")

### Your turn 4 (3 minutes)

**Predict first:** which two notes come back for "oat"? Then add a note of your own (use words the vocabulary knows) and query for it.

In [ ]:
vocab, E, store = lucy_store()
print(store.query(["oat"], n_results=2)["ids"])
store.add(ids=["mine"], documents=["kids want mango sorbet"])
print(store.query(["sorbet"], n_results=3)["ids"])

## Part 4b · The same with Chroma, on Colab

Now the real thing: [Chroma](https://docs.trychroma.com/) stores the same twelve notes and embeds them with a trained model — **hundreds of numbers per vector**, not our two.

**Your OpenAI key.** In Colab, open the key icon in the left sidebar, add a secret named `OPENAI_API_KEY`, and switch on notebook access for it. This notebook never prints the key. With a key, Chroma embeds with OpenAI's `text-embedding-3-small`. Without one, it uses Chroma's default local model, `all-MiniLM-L6-v2`, which downloads once. Both runs are labeled.

Install Chroma (Colab only, about a minute):

In [ ]:
%pip install -q chromadb

In [ ]:
import os

def resolve_api_key():
    """The Colab secret first, then the environment. Never raises and never prints the key."""
    try:
        from google.colab import userdata
        value = userdata.get("OPENAI_API_KEY")
        if value:
            return str(value).strip()
    except Exception:
        pass
    value = os.environ.get("OPENAI_API_KEY")
    return value.strip() if value else None

import chromadb
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction, OpenAIEmbeddingFunction

api_key = resolve_api_key()
if api_key:
    os.environ["OPENAI_API_KEY"] = api_key  # Chroma's OpenAI embedding function reads this variable
    embedding_function = OpenAIEmbeddingFunction(model_name="text-embedding-3-small")
    EMBEDDINGS = "live: OpenAI text-embedding-3-small"
else:
    embedding_function = DefaultEmbeddingFunction()
    EMBEDDINGS = "offline: Chroma default all-MiniLM-L6-v2"
print("Embeddings:", EMBEDDINGS, "| key found:", api_key is not None)

client = chromadb.Client()
try:
    client.delete_collection("lucy_notes")  # lets you re-run this cell
except Exception:
    pass
collection = client.create_collection(
    name="lucy_notes",
    embedding_function=embedding_function,
    configuration={"hnsw": {"space": "cosine"}},
)
collection.add(
    ids=[f"n{i + 1}" for i in range(len(NOTES))],
    documents=NOTES,
    metadatas=[{"topic": "menu" if i < 8 else "supplier", "current": True} for i in range(len(NOTES))],
)
print(collection.count(), "notes stored")

**Prediction:** with a trained model, which four notes come back for "a vegan treat" — the same four as our two-number vectors?

In [ ]:
results = collection.query(query_texts=["a vegan treat"], n_results=4)
for note_id, document, distance in zip(results["ids"][0], results["documents"][0], results["distances"][0]):
    print(f"{note_id:>4}  cosine distance {distance:.3f}  {document}")
_, _, scratch = lucy_store(space="cosine")  # a clean copy, without your turn-4 note
print("from scratch:", scratch.query(["a vegan treat"], n_results=4)["ids"][0])
print("Chroma      :", results["ids"][0], "|", EMBEDDINGS)
assert len(results["ids"][0]) == 4

How many numbers per vector does a real model use?

In [ ]:
row = collection.get(ids=["n1"], include=["embeddings"])
print(len(row["embeddings"][0]), "numbers per vector |", EMBEDDINGS)
print("ours:", len(scratch.rows["n1"]["embedding"]), "numbers per vector")

The stale note, in Chroma: add the Friday note, see what ranks first, then filter on metadata and delete. **Prediction:** does a trained model tell Monday from Friday better than ours?

In [ ]:
collection.add(ids=["n13"], documents=[NEW_DELIVERY], metadatas=[{"topic": "supplier", "current": True}])
query = ["supplier delivers tubs"]
print("nearest two:", collection.query(query_texts=query, n_results=2)["documents"][0])
collection.update(ids=["n9"], metadatas=[{"topic": "supplier", "current": False}])
print("where current=True:", collection.query(query_texts=query, n_results=1, where={"current": True})["documents"][0])
collection.delete(ids=["n9"])
print("after delete:", collection.query(query_texts=query, n_results=1)["documents"][0])
assert "n9" not in collection.get()["ids"]

**Your turn (open):** add three notes of your own to `collection` and query them. Unlike our lab, Chroma's model knows words that never appeared in Lucy's notes — try "a treat without milk".

## Keep building with Prof Rod

Found this material through a colleague, classroom or shared download? [Get the complete book at profrod.ai/book](https://profrod.ai/book) and [join the Prof Rod learner community](https://profrod.ai/community). Bring one result, one question or one failure you learned from. Share this resource with another learner and keep its source links with it so they can find the full course and future updates.